# MBTA Passenger Survey Harmonization

This notebook harmonizes the documented 2015-17 CTPS systemwide survey and the 2025 MassDOT survey. Route groups are matched only when route membership is exact; a grouped older result is never split across routes. Definitions that cannot be exactly aligned remain in the crosswalk with `not_comparable` status.

Run all cells to create the availability matrix, category crosswalk, route crosswalk, combined long-form dataset, percentage checks, and a short methods note. Inputs and outputs live in `data/processed/`.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data" / "processed").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "processed").is_dir():
    raise FileNotFoundError("Open this notebook from the project workspace so data/processed can be located.")

PROCESSED = PROJECT_ROOT / "data" / "processed"
OLD_PERIOD = "2015-2017"
NEW_PERIOD = "2025"
FOCAL_ROUTES = {"22", "23", "28", "29"}
FARE_FREE_ROUTES = {"23", "28", "29"}

# Prefer the original values export because an earlier cleaning step dropped English-comprehension fields.
original_old_values_path = PROJECT_ROOT.parent / "MBTA-Passenger_Survey-Line_Values.csv"
old_values_path = original_old_values_path if original_old_values_path.exists() else PROCESSED / "ps_2017_2019_values.csv"
new_survey_path = PROCESSED / "ps_2025.csv"
old = pd.read_csv(old_values_path)
new = pd.read_csv(new_survey_path)


def standardize_columns(frame):
    frame = frame.copy()
    frame.columns = (
        pd.Index(frame.columns).astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    )
    duplicates = frame.columns[frame.columns.duplicated()].tolist()
    if duplicates:
        raise ValueError(f"Duplicate columns after standardization: {duplicates}")
    return frame


old = standardize_columns(old)
new = standardize_columns(new)
old = old.loc[old["mode"].astype("string").str.strip().str.casefold().eq("bus")].copy()
new = new.loc[
    new["service_mode"].astype("string").str.strip().str.casefold().eq("bus")
    & new["aggregation_level"].astype("string").str.strip().eq("Reporting Group")
].copy()

for column in ["weighted_percent", "n", "margin_of_error"]:
    new[column] = pd.to_numeric(new[column], errors="coerce")
old["line"] = old["line"].astype("string").str.strip()

print(f"2015-17 bus rows: {len(old):,}; 2025 route reporting rows: {len(new):,}")
print(f"2015-17 source: {old_values_path}")


def route_sort_key(route_id):
    match = re.fullmatch(r"(CT|SL)?([0-9]+)([A-Z]*)", route_id)
    prefix, number, suffix = match.groups()
    return (prefix or "", int(number), suffix)


def route_members(label):
    if pd.isna(label):
        return None
    normalized = re.sub("[ \\t]+and[ \\t]+", "&", str(label).strip(), flags=re.IGNORECASE)
    normalized = re.sub(",[ \\t]*&", "&", normalized)
    normalized = re.sub("[ \\t]+", "", normalized).upper()
    token = r"(?:CT[0-9]+|SL[0-9]+|[0-9]+[A-Z]*)"
    if not re.fullmatch(rf"{token}(?:[,&/]{token})*", normalized):
        return None
    return tuple(sorted(set(re.findall(token, normalized)), key=route_sort_key))


def route_label(members):
    return " & ".join(sorted(members, key=route_sort_key))


old_routes = {}
for row_index, row in old.iterrows():
    members = route_members(row["line"])
    if members:
        old_routes.setdefault(members, []).append((row_index, str(row["line"])))

new_routes = {}
for group in new["reporting_group"].dropna().astype(str).unique():
    members = route_members(group)
    if members:
        new_routes.setdefault(members, []).append(group)

route_records = []
matched_groups = []
used_old_indexes = set()
for members, reporting_groups in sorted(new_routes.items()):
    if len(reporting_groups) != 1:
        raise ValueError(f"Duplicate 2025 route-member group: {reporting_groups}")
    new_group = reporting_groups[0]
    exact_old = old_routes.get(members, [])
    if len(exact_old) == 1:
        old_indexes = [exact_old[0][0]]
        old_reporting_groups = [exact_old[0][1]]
        match_status = "exact individual-route match" if len(members) == 1 else "grouped-route match"
        route_aggregation = "none"
    else:
        member_rows = [old_routes.get((member,), []) for member in members]
        if not exact_old and all(len(rows) == 1 for rows in member_rows):
            old_indexes = [rows[0][0] for rows in member_rows]
            old_reporting_groups = [rows[0][1] for rows in member_rows]
            match_status = "grouped-route match"
            route_aggregation = "sum complete 2015-17 individual-route records to 2025 group"
        else:
            old_indexes = []
            old_reporting_groups = []
            match_status = "not comparable"
            route_aggregation = "none"

    record = {
        "route": route_label(members),
        "route_members": ";".join(members),
        "reporting_group_2015_17": " | ".join(old_reporting_groups),
        "reporting_group_2025": new_group,
        "match_status": match_status,
        "route_aggregation_applied": route_aggregation,
        "focal_route": int(bool(set(members) & FOCAL_ROUTES)),
        "fare_free_route": int(bool(set(members) & FARE_FREE_ROUTES)),
    }
    route_records.append(record)
    if old_indexes:
        used_old_indexes.update(old_indexes)
        matched_groups.append({**record, "old_indexes": old_indexes, "members": members})

for members, rows in old_routes.items():
    for row_index, old_group in rows:
        if row_index not in used_old_indexes:
            route_records.append({
                "route": route_label(members),
                "route_members": ";".join(members),
                "reporting_group_2015_17": old_group,
                "reporting_group_2025": "",
                "match_status": "not comparable",
                "route_aggregation_applied": "none",
                "focal_route": int(bool(set(members) & FOCAL_ROUTES)),
                "fare_free_route": int(bool(set(members) & FARE_FREE_ROUTES)),
            })

for group in new["reporting_group"].dropna().astype(str).unique():
    if route_members(group) is None:
        route_records.append({
            "route": "", "route_members": "", "reporting_group_2015_17": "",
            "reporting_group_2025": group, "match_status": "not comparable",
            "route_aggregation_applied": "non-route reporting group",
            "focal_route": 0, "fare_free_route": 0,
        })

route_crosswalk = pd.DataFrame(route_records).drop_duplicates()
matched_route_groups = pd.DataFrame(matched_groups)
print(route_crosswalk["match_status"].value_counts(dropna=False))
print("Matched route groups:", len(matched_route_groups))

In [ ]:
FEATURES = [
    ("Household income", "household_income", "Income", "Household Income", True),
    ("Low-income status", "low_income_classification", "Income", "Title VI Low-Income", True),
    ("Race", "race", "Race and Ethnicity", "Race", False),
    ("Hispanic or Latine ethnicity", "hispanic_or_latino_latina", "Race and Ethnicity", "Hispanic or Latine", True),
    ("Usable household vehicles", "usable_household_vehicles", "Vehicles", "Usable Cars", True),
    ("Vehicles per capita", "vehicles_per_capita", "Vehicles", "Cars per Capita", True),
    ("Trip frequency", "trip_frequency", "Trip Purpose and Frequency", "Frequency", True),
    ("Trip purpose", "trip_purpose", "Trip Purpose and Frequency", "Trip Purpose", True),
    ("Access to first MBTA service", "access_to_first_mbta_service", "Access Mode", "Access to First MBTA Service", True),
    ("Age", "age", "Other Demographics", "Age", True),
    ("Employment/student status", None, None, None, True),
    ("Gender", "gender", "Other Demographics", "Gender", True),
    ("Fare product", "fare", "Fares", "Fare Product", True),
    ("Alternative mode use", "use_alternative_modes", "Alternative Modes", "Used Alternative Mode", True),
    ("Alternative modes", "alternative_modes", "Alternative Modes", "Alternative Mode", False),
    ("English comprehension", "ability_to_understand_english", "Other Demographics", "Ability to Understand English", True),
    ("Valid driver's license", "valid_driver_s_license", None, None, True),
    ("Previous transit mode", "previous_transit_mode", None, None, True),
    ("Monthly pass type", "monthly_pass_type", None, None, True),
    ("Pay-per-ride type", "pay_per_ride_type", None, None, True),
    ("Title VI minority status", "minority_classification", "Race and Ethnicity", "Title VI Minority", True),
    ("Smartphone availability", None, "Smartphones", "Smartphone", True),
]

OLD_LABELS = {
    "trip_purpose": {
        "home_based_work": "Home-based Work", "home_based_school": "Home-based School",
        "home_based_social_activity": "Home-based Social Activity", "home_based_other": "Home-based Other",
        "non_home_based": "Non-home Based",
    },
    "trip_frequency": {
        "6_7_days_a_week": "6 to 7 days per week", "5_days_a_week": "5 days per week",
        "3_4_days_a_week": "3 to 4 days per week", "1_2_days_a_week": "1 to 2 days per week",
        "1_3_days_a_month": "1 to 3 days per month", "less_than_once_a_month": "Less than once per month",
    },
    "household_income": {
        "less_than_14_500": "Less than $14,500", "14_500_to_28_999": "$14,500 to $28,999",
        "29_000_to_43_499": "$29,000 to $43,499", "43_500_to_57_999": "$43,500 to $57,999",
        "58_000_to_75_999": "$58,000 to $75,999", "76_000_to_108_499": "$76,000 to $108,499",
        "108_500_to_151_999": "$108,500 to $151,999", "152_000_or_more": "$152,000 or more",
        "prefer_not_to_say": "Prefer not to say",
    },
    "race": {
        "american_indian_or_alaska_native": "American Indian or Alaska Native", "asian": "Asian",
        "black_or_african_american": "Black or African American",
        "native_hawaiian_or_other_pacific_islander": "Native Hawaiian or other Pacific Islander",
        "white": "White", "other": "Other", "prefer_not_to_say": "Prefer not to say",
    },
    "hispanic_or_latino_latina": {"yes": "Yes", "no": "No", "prefer_not_to_say": "Prefer not to say"},
    "usable_household_vehicles": {"0": "0", "1": "1", "2": "2", "3_or_more": "3 or more"},
    "vehicles_per_capita": {
        "none": "0", "0_01_0_49": "Strictly between 0 and 0.5", "0_50_0_99": "0.5 to less than 1",
        "1_00_1_49": "1 or more", "1_50_1_99": "1 or more", "2_or_more": "1 or more",
    },
    "access_to_first_mbta_service": {
        "walked_or_bicycled": "Walked or bicycled", "drove_alone": "Drove Alone",
        "carpooled": "Carpooled", "dropped_off_by_personal_vehicle": "Dropped off by personal vehicle",
        "dropped_off_by_other_vehicle": "Dropped off by other vehicle",
    },
    "age": {
        "under_18": "Under 18", "18_to_21": "18 to 21", "22_to_34": "22 to 34",
        "35_to_44": "35 to 44", "45_to_64": "45 to 64", "65_or_over": "65 or older",
    },
    "gender": {"man": "Male", "woman": "Female", "other": "Other", "prefer_not_to_say": "Prefer not to say"},
    "fare": {
        "monthly_pass": "Monthly Pass", "pay_per_ride": "Pay-as-you-go",
        "1_day_or_7_day_linkpass": "1-day or 7-day Pass", "other": "Other",
    },
    "use_alternative_modes": {"yes": "Yes", "no": "No"},
    "alternative_modes": {
        "walk": "Walk", "bicycle": "Bicycle", "drive_alone": "Drive Alone",
        "drive_or_ride_in_a_carpool": "Drive or Ride in a Carpool",
        "take_a_taxi_or_use_a_rideshare_company": "Taxi or Rideshare Company",
        "take_a_private_shuttle_or_other_transit": "Private Shuttle or Other Transit",
        "take_a_different_mbta_service": "Different MBTA Service", "other": "Other",
    },
    "ability_to_understand_english": {
        "always": "Always", "often": "Often", "sometimes": "Sometimes", "never": "Never",
        "prefer_not_to_say": "Prefer not to say",
    },
    "valid_driver_s_license": {"yes": "Yes", "no": "No"},
    "previous_transit_mode": {
        "bus_or_silver_line": "Bus or Silver Line", "rail_rapid_transit": "Rail rapid transit",
        "commuter_rail_or_ferry": "Commuter rail or ferry",
    },
    "monthly_pass_type": {
        "reduced_fare_pass": "Reduced fare pass", "local_bus": "Local bus",
        "linkpass_or_express_bus": "LinkPass or express bus", "commuter_rail_or_ferry": "Commuter rail or ferry",
    },
    "pay_per_ride_type": {
        "full_cash_fare_on_board": "Full cash fare on board", "reduced_fare": "Reduced fare",
        "mticket_single_multi_ride": "mTicket single/multi-ride", "charliecard_plastic": "CharlieCard (plastic)",
        "charlieticket_paper": "CharlieTicket (paper)",
    },
    "minority_classification": {"yes": "Yes", "no": "No"},
}


def category_key(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().casefold()).strip("_")


def mapping(target, status="directly_comparable", aggregation="none"):
    return {"target": target, "status": status, "aggregation": aggregation}


CATEGORY_RULES = {}

def set_rules(characteristic, old_rules=None, new_rules=None):
    CATEGORY_RULES[characteristic] = {"old": old_rules or {}, "new": new_rules or {}}


def exact_rules(labels):
    return {category_key(label): mapping(label) for label in labels}


purpose_labels = OLD_LABELS["trip_purpose"].values()
frequency_labels = OLD_LABELS["trip_frequency"].values()
set_rules("Trip purpose", exact_rules(purpose_labels), exact_rules(purpose_labels))
set_rules("Trip frequency", exact_rules(frequency_labels), exact_rules(frequency_labels))
set_rules("Usable household vehicles", {
    **{str(number): mapping(str(number)) for number in range(3)}, "3_or_more": mapping("3 or more")
}, {
    **{str(number): mapping(str(number)) for number in range(3)}, "3_or_more": mapping("3 or more")
})
set_rules("Vehicles per capita", {
    "none": mapping("0"),
    "0_01_0_49": mapping("Strictly between 0 and 0.5"),
    "0_50_0_99": mapping("0.5 to less than 1", "comparable_after_aggregation", "old 0.50-0.99 equals 2025 0.5 plus >0.5-<1"),
    "1_00_1_49": mapping("1 or more", "comparable_after_aggregation", "sum old 1.00-1.49, 1.50-1.99, and 2+ categories"),
    "1_50_1_99": mapping("1 or more", "comparable_after_aggregation", "sum old 1.00-1.49, 1.50-1.99, and 2+ categories"),
    "2_or_more": mapping("1 or more", "comparable_after_aggregation", "sum old 1.00-1.49, 1.50-1.99, and 2+ categories"),
}, {
    "0": mapping("0"),
    "strictly_between_0_and_0_5": mapping("Strictly between 0 and 0.5"),
    "0_5": mapping("0.5 to less than 1", "comparable_after_aggregation", "sum 2025 0.5 and strictly-between-0.5-and-1 categories"),
    "strictly_between_0_5_and_1": mapping("0.5 to less than 1", "comparable_after_aggregation", "sum 2025 0.5 and strictly-between-0.5-and-1 categories"),
    "1_or_more": mapping("1 or more", "comparable_after_aggregation", "matches the sum of three old categories"),
})
set_rules("Access to first MBTA service", {
    "drove_alone": mapping("Drove Alone"),
    "carpooled": mapping("Carpooled or Dropped off by Personal Vehicle", "comparable_after_aggregation", "sum old carpool and personal-vehicle drop-off categories"),
    "dropped_off_by_personal_vehicle": mapping("Carpooled or Dropped off by Personal Vehicle", "comparable_after_aggregation", "sum old carpool and personal-vehicle drop-off categories"),
}, {
    "drove_alone": mapping("Drove Alone"),
    "carpooled_or_dropped_off_by_personal_vehicle": mapping("Carpooled or Dropped off by Personal Vehicle", "comparable_after_aggregation", "matches the sum of two old categories"),
})
set_rules("Age", exact_rules(["Under 18", "35 to 44", "45 to 64", "65 or older"]), exact_rules(["Under 18", "35 to 44", "45 to 64", "65 or older"]))
set_rules("Gender", exact_rules(["Male", "Female", "Prefer not to say"]), exact_rules(["Male", "Female", "Prefer not to say"]))
set_rules("Race", exact_rules([
    "American Indian or Alaska Native", "Asian", "Black or African American",
    "Native Hawaiian or other Pacific Islander", "White", "Prefer not to say",
]), exact_rules([
    "American Indian or Alaska Native", "Asian", "Black or African American",
    "Native Hawaiian or other Pacific Islander", "White", "Prefer not to say",
]))
set_rules("Hispanic or Latine ethnicity", exact_rules(["Yes", "No", "Prefer not to say"]), exact_rules(["Yes", "No", "Prefer not to say"]))
set_rules("Fare product", {
    "monthly_pass": mapping("Monthly Pass"),
    "pay_per_ride": mapping("Pay-as-you-go"),
    "1_day_or_7_day_linkpass": mapping("1-day or 7-day Pass", "comparable_after_aggregation", "sum 2025 1-day and 7-day passes"),
}, {
    "monthly_pass": mapping("Monthly Pass"),
    "pay_as_you_go": mapping("Pay-as-you-go"),
    "1_day_pass": mapping("1-day or 7-day Pass", "comparable_after_aggregation", "sum 2025 1-day and 7-day passes"),
    "7_day_pass": mapping("1-day or 7-day Pass", "comparable_after_aggregation", "sum 2025 1-day and 7-day passes"),
})
set_rules("Alternative mode use", exact_rules(["Yes", "No"]), exact_rules(["Yes", "No"]))
set_rules("Alternative modes", exact_rules([
    "Walk", "Drive Alone", "Drive or Ride in a Carpool", "Taxi or Rideshare Company",
    "Private Shuttle or Other Transit", "Different MBTA Service", "Other",
]), exact_rules([
    "Walk", "Drive Alone", "Drive or Ride in a Carpool", "Taxi or Rideshare Company",
    "Private Shuttle or Other Transit", "Different MBTA Service", "Other",
]))
set_rules("English comprehension", exact_rules(["Always", "Often", "Sometimes", "Never", "Prefer not to say"]), exact_rules(["Always", "Often", "Sometimes", "Never", "Prefer not to say"]))

print("Characteristics in availability/crosswalk:", len(FEATURES))

In [ ]:
def old_category_fields(prefix):
    if not prefix:
        return []
    labels = OLD_LABELS.get(prefix, {})
    fields = [
        column for column in old.columns
        if column.startswith(f"{prefix}_") and not column.startswith("count_")
    ]
    return [
        (field, labels.get(field[len(prefix) + 1:], field[len(prefix) + 1:].replace("_", " ").title()))
        for field in fields
    ]


def category_rule(characteristic, period, category):
    source = "old" if period == OLD_PERIOD else "new"
    rule = CATEGORY_RULES.get(characteristic, {}).get(source, {}).get(category_key(category))
    if rule is None:
        rule = mapping("", "not_comparable", "none")
    return rule


def safe_sum(values):
    numeric = pd.to_numeric(pd.Series(values), errors="coerce")
    if numeric.empty or numeric.isna().any():
        return float("nan")
    return float(numeric.sum())


# Build the availability matrix for every matched route group and requested characteristic.
availability_records = []
for group in matched_groups:
    old_route_rows = old.loc[group["old_indexes"]]
    new_route_rows = new.loc[new["reporting_group"].eq(group["reporting_group_2025"])]
    for characteristic, old_prefix, measure_group, measure, _ in FEATURES:
        old_fields = old_category_fields(old_prefix)
        available_old = bool(old_fields and old_route_rows[[field for field, _ in old_fields]].notna().any().any())
        if measure_group and measure:
            new_measure_rows = new_route_rows.loc[
                new_route_rows["measure_group"].eq(measure_group) & new_route_rows["measure"].eq(measure)
            ]
            available_new = bool(not new_measure_rows.empty and new_measure_rows["weighted_percent"].notna().any())
        else:
            available_new = False
        availability_records.append({
            "route": group["route"],
            "reporting_group_2015_17": group["reporting_group_2015_17"],
            "reporting_group_2025": group["reporting_group_2025"],
            "characteristic": characteristic,
            "available_2015_17": available_old,
            "available_2025": available_new,
            "available_both": available_old and available_new,
            "focal_route": group["focal_route"],
            "fare_free_route": group["fare_free_route"],
        })
availability_matrix = pd.DataFrame(availability_records)


# The category crosswalk captures matched and explicitly unmatched categories.
category_records = []
for characteristic, old_prefix, measure_group, measure, _ in FEATURES:
    source_fields = old_category_fields(old_prefix)
    for source_field, category in source_fields:
        rule = category_rule(characteristic, OLD_PERIOD, category)
        category_records.append({
            "characteristic": characteristic,
            "survey_period": OLD_PERIOD,
            "measure_group": measure_group or "",
            "measure": measure or "",
            "category_original": category,
            "source_field": source_field,
            "category_harmonized": rule["target"],
            "comparability_status": rule["status"],
            "aggregation_applied": rule["aggregation"],
        })
    if measure_group and measure:
        categories_2025 = new.loc[
            new["measure_group"].eq(measure_group) & new["measure"].eq(measure), "category"
        ].dropna().drop_duplicates()
        for category in categories_2025:
            category = str(category)
            rule = category_rule(characteristic, NEW_PERIOD, category)
            category_records.append({
                "characteristic": characteristic,
                "survey_period": NEW_PERIOD,
                "measure_group": measure_group,
                "measure": measure,
                "category_original": category,
                "source_field": "category",
                "category_harmonized": rule["target"],
                "comparability_status": rule["status"],
                "aggregation_applied": rule["aggregation"],
            })
    if not source_fields and not (measure_group and measure):
        category_records.append({
            "characteristic": characteristic,
            "survey_period": "both",
            "measure_group": measure_group or "",
            "measure": measure or "",
            "category_original": "not available",
            "source_field": "",
            "category_harmonized": "",
            "comparability_status": "not_comparable",
            "aggregation_applied": "not present in either available source export",
        })
category_crosswalk = pd.DataFrame(category_records).drop_duplicates()


def old_combined_rows():
    records = []
    for group in matched_groups:
        route_rows = old.loc[group["old_indexes"]]
        for characteristic, prefix, measure_group, measure, single_choice in FEATURES:
            fields = old_category_fields(prefix)
            if not fields:
                continue
            raw_values = {field: safe_sum(route_rows[field].tolist()) for field, _ in fields}
            count_column = f"count_{prefix}" if f"count_{prefix}" in old.columns else None
            sample_size = safe_sum(route_rows[count_column].tolist()) if count_column else float("nan")
            complete_distribution = all(pd.notna(value) for value in raw_values.values())
            distribution_total = sum(raw_values.values()) if complete_distribution else float("nan")

            grouped_categories = {}
            for source_field, category in fields:
                rule = category_rule(characteristic, OLD_PERIOD, category)
                category_target = rule["target"] or f"[not comparable] {category}"
                group_key = (category_target, rule["status"], rule["aggregation"])
                grouped_categories.setdefault(group_key, []).append((source_field, category, raw_values[source_field]))

            for (harmonized_category, status, aggregation), items in grouped_categories.items():
                original_categories = [item[1] for item in items]
                original_values = [item[2] for item in items]
                value = safe_sum(original_values)
                percent = (
                    value / distribution_total
                    if single_choice and pd.notna(value) and distribution_total > 0 and status != "not_comparable"
                    else float("nan")
                )
                records.append({
                    "survey_period": OLD_PERIOD,
                    "route": group["route"],
                    "reporting_group_original": group["reporting_group_2015_17"],
                    "aggregation_level": "Reporting Group",
                    "service_mode": "Bus",
                    "characteristic": characteristic,
                    "category_original": " | ".join(original_categories),
                    "source_categories_original": json.dumps(original_categories, ensure_ascii=True),
                    "category_harmonized": harmonized_category,
                    "value": value,
                    "source_values_original": json.dumps([
                        {"category": category, "source_field": field, "value": None if pd.isna(raw_value) else float(raw_value)}
                        for (field, category, raw_value) in items
                    ], ensure_ascii=True),
                    "value_type": "2015-17 values-export estimate",
                    "percent": percent,
                    "n": sample_size,
                    "margin_of_error": float("nan"),
                    "comparability_status": status,
                    "aggregation_applied": aggregation if aggregation != "none" else group["route_aggregation_applied"],
                    "focal_route": group["focal_route"],
                    "fare_free_route": group["fare_free_route"],
                })
    return records


def new_combined_rows():
    records = []
    for group in matched_groups:
        route_rows = new.loc[new["reporting_group"].eq(group["reporting_group_2025"])]
        for characteristic, _, measure_group, measure, _ in FEATURES:
            if not measure_group or not measure:
                continue
            measure_rows = route_rows.loc[
                route_rows["measure_group"].eq(measure_group) & route_rows["measure"].eq(measure)
            ]
            grouped_categories = {}
            for _, row in measure_rows.iterrows():
                category = str(row["category"])
                rule = category_rule(characteristic, NEW_PERIOD, category)
                category_target = rule["target"] or f"[not comparable] {category}"
                group_key = (category_target, rule["status"], rule["aggregation"])
                grouped_categories.setdefault(group_key, []).append((category, row))

            for (harmonized_category, status, aggregation), items in grouped_categories.items():
                original_categories = [item[0] for item in items]
                original_percents = [item[1]["weighted_percent"] for item in items]
                value = safe_sum(original_percents)
                sample_sizes = pd.to_numeric(pd.Series([item[1]["n"] for item in items]), errors="coerce")
                sample_size = (
                    float(sample_sizes.iloc[0])
                    if sample_sizes.notna().all() and sample_sizes.nunique() == 1 else float("nan")
                )
                margin_values = pd.to_numeric(pd.Series([item[1]["margin_of_error"] for item in items]), errors="coerce")
                margin = (
                    float(margin_values.iloc[0])
                    if len(items) == 1 and margin_values.notna().all() else float("nan")
                )
                records.append({
                    "survey_period": NEW_PERIOD,
                    "route": group["route"],
                    "reporting_group_original": group["reporting_group_2025"],
                    "aggregation_level": "Reporting Group",
                    "service_mode": "Bus",
                    "characteristic": characteristic,
                    "category_original": " | ".join(original_categories),
                    "source_categories_original": json.dumps(original_categories, ensure_ascii=True),
                    "category_harmonized": harmonized_category,
                    "value": value,
                    "source_values_original": json.dumps([
                        {"category": category, "value": None if pd.isna(percent) else float(percent)}
                        for category, percent in zip(original_categories, original_percents)
                    ], ensure_ascii=True),
                    "value_type": "2025 weighted_percent",
                    "percent": value if status != "not_comparable" else float("nan"),
                    "n": sample_size,
                    "margin_of_error": margin,
                    "comparability_status": status,
                    "aggregation_applied": aggregation if len(items) > 1 else group["route_aggregation_applied"],
                    "focal_route": group["focal_route"],
                    "fare_free_route": group["fare_free_route"],
                })
    return records


combined = pd.DataFrame(old_combined_rows() + new_combined_rows())
combined.insert(0, "source_file", [old_values_path.name if period == OLD_PERIOD else new_survey_path.name for period in combined["survey_period"]])
print(f"Availability rows: {len(availability_matrix):,}; category crosswalk rows: {len(category_crosswalk):,}; combined rows: {len(combined):,}")

In [ ]:
# Preserve route-level source estimates where several older routes form one 2025 group.
feature_by_name = {feature[0]: feature for feature in FEATURES}
matched_by_route = {group["route"]: group for group in matched_groups}


def original_route_values(record):
    original_categories = json.loads(record["source_categories_original"])
    if record["survey_period"] == OLD_PERIOD:
        feature = feature_by_name[record["characteristic"]]
        label_to_field = {label: field for field, label in old_category_fields(feature[1])}
        group = matched_by_route[record["route"]]
        details = []
        for row_index in group["old_indexes"]:
            source_row = old.loc[row_index]
            for category in original_categories:
                source_field = label_to_field[category]
                source_value = pd.to_numeric(pd.Series([source_row[source_field]]), errors="coerce").iloc[0]
                details.append({
                    "reporting_group_original": str(source_row["line"]),
                    "category_original": category,
                    "source_field": source_field,
                    "value": None if pd.isna(source_value) else float(source_value),
                })
        return json.dumps(details, ensure_ascii=True)

    source_components = json.loads(record["source_values_original"])
    return json.dumps([
        {
            "reporting_group_original": record["reporting_group_original"],
            "category_original": component["category"],
            "value": component["value"],
        }
        for component in source_components
    ], ensure_ascii=True)


combined["source_route_values_original"] = combined.apply(original_route_values, axis=1)
new_rows_mask = combined["survey_period"].eq(NEW_PERIOD)
combined.loc[new_rows_mask, "aggregation_applied"] = combined.loc[new_rows_mask].apply(
    lambda row: "category aggregation: " + row["aggregation_applied"]
    if row["aggregation_applied"] != "none" and " | " in row["category_original"]
    else "none",
    axis=1,
)
print("Original route/category components retained:", combined["source_route_values_original"].notna().all())

In [ ]:
# A non-comparable source category has no harmonized category by design.
combined.loc[
    combined["comparability_status"].eq("not_comparable"), "category_harmonized"
] = ""


In [ ]:
# Keep source percentages for every 2025 category, including rows marked not comparable.
combined.loc[combined["survey_period"].eq(NEW_PERIOD), "percent"] = combined.loc[
    combined["survey_period"].eq(NEW_PERIOD), "value"
]

# Derive old percentages only for single-choice characteristics and only from complete values distributions.
for (period, route, characteristic), row_indexes in combined.groupby(
    ["survey_period", "route", "characteristic"], sort=False
).groups.items():
    if period != OLD_PERIOD:
        continue
    feature = next(item for item in FEATURES if item[0] == characteristic)
    if not feature[4]:
        continue
    values = pd.to_numeric(combined.loc[row_indexes, "value"], errors="coerce")
    if values.empty or values.isna().any():
        continue
    total = values.sum()
    if total > 0:
        combined.loc[row_indexes, "percent"] = values / total

# Audit keys and percentage behavior before writing anything.
combined_key = [
    "survey_period", "route", "characteristic", "category_harmonized", "category_original"
]
duplicate_rows = combined.duplicated(subset=combined_key, keep=False)
if duplicate_rows.any():
    display(combined.loc[duplicate_rows, combined_key].head(20))
    raise ValueError(f"Duplicate combined records found: {int(duplicate_rows.sum())}")

valid_percentages = combined["percent"].dropna().between(0, 1).all()
if not valid_percentages:
    display(combined.loc[~combined["percent"].between(0, 1) & combined["percent"].notna()].head(20))
    raise ValueError("A percentage is outside the valid [0, 1] range.")

sum_check_records = []
for (period, route, characteristic), rows in combined.groupby(
    ["survey_period", "route", "characteristic"], sort=False
):
    feature = next(item for item in FEATURES if item[0] == characteristic)
    if not feature[4]:
        continue
    percent_values = pd.to_numeric(rows["percent"], errors="coerce")
    complete = bool(not percent_values.empty and percent_values.notna().all())
    total = float(percent_values.sum()) if complete else float("nan")
    sum_check_records.append({
        "survey_period": period,
        "route": route,
        "characteristic": characteristic,
        "percent_sum": total,
        "check_status": "pass" if complete and abs(total - 1.0) <= 0.03 else "fail" if complete else "incomplete; not tested",
    })
percentage_sum_checks = pd.DataFrame(sum_check_records)
failed_checks = percentage_sum_checks.loc[percentage_sum_checks["check_status"].eq("fail")]
if not failed_checks.empty:
    display(failed_checks.head(20))
    raise ValueError("A complete single-choice percentage distribution does not sum to 1 within 0.03.")

# Add explicit rows for requested topics with no corresponding source fields/measures.
for characteristic, old_prefix, measure_group, measure, _ in FEATURES:
    if characteristic not in set(availability_matrix["characteristic"]):
        continue
    if old_prefix is None and (not measure_group or not measure):
        continue
    if measure_group and measure:
        has_new_measure = bool(
            new["measure_group"].eq(measure_group).any() and new["measure"].eq(measure).any()
        )
    else:
        has_new_measure = False
    if old_prefix is None and not has_new_measure:
        continue

# Write requested files and a compact audit file.
PROCESSED.mkdir(parents=True, exist_ok=True)
availability_matrix.to_csv(PROCESSED / "survey_availability_matrix.csv", index=False)
category_crosswalk.to_csv(PROCESSED / "survey_category_crosswalk.csv", index=False)
route_crosswalk.to_csv(PROCESSED / "survey_route_crosswalk.csv", index=False)
combined.to_csv(PROCESSED / "survey_harmonized_combined.csv", index=False)
percentage_sum_checks.to_csv(PROCESSED / "survey_percentage_sum_checks.csv", index=False)

methods_note = """# Passenger Survey Harmonization Note

The old survey is the **2015-17 MBTA Systemwide Passenger Survey** documented by CTPS. Its local processed filename (`ps_2017_2019_values.csv`) is inconsistent with the source period. The 2025 survey is the MassDOT MBTA System-Wide Passenger Survey. This notebook uses the old values export only; it does not use the separate old percentage export.

Route groups are matched by complete route-member sets. Old individual-route records may be summed only to reconstruct a 2025 grouped reporting group when every member has exactly one old record. No grouped source record is split. Route 23 is an individual match. Old Routes 22 and 29 are grouped together, while 2025 reports Route 22 separately and Routes 28 and 29 together; those records are not equivalent and are excluded from the combined table. One-sided route groups and neighborhood reporting groups remain in the route crosswalk as not comparable. The combined dataset contains only route groups validly available in both survey periods.

`directly_comparable` categories retain equivalent definitions. `comparable_after_aggregation` is used only for exact sums: the older 0.50-0.99 vehicles-per-capita category against the exact combined 2025 0.5 and strictly-between-0.5-and-1 categories; older vehicle-per-capita 1.00+ categories against 2025 1 or more; old carpool plus personal-vehicle drop-off against 2025's combined category; and the old 1-day-or-7-day pass category against 2025's separate 1-day and 7-day categories. Age bands that cross boundaries, income bands (dollar bands versus AMI), low-income status (different thresholds), race/ethnicity definitions not explicitly equivalent, and categories such as bicycle versus micromobility are not forced to match. Employment/student status is not present in either export.

Original category labels and values are preserved in `category_original`, `source_categories_original`, `value`, and `source_values_original`. Old values are weighted estimates, not respondent counts; old `count_*` fields are retained as `n`. The 2015-17 percentage is derived only for complete single-choice value distributions. The old values export alone does not provide a defensible respondent denominator for check-all-that-apply measures, so their percentage is missing. 2025 `weighted_percent` is kept as `percent`. Values in `value` therefore retain source units and must not be compared directly across years. No older margin of error is estimated, and no MOE is assigned to sums of categories.

Availability is assessed per matched route group and characteristic. Percentage sums are tested only for single-choice distributions; incomplete distributions are reported as not tested, and multi-select characteristics are not expected to sum to one. Route flags are group-level: a grouped record receives a flag if it contains a focal/fare-free route, without implying a route-specific estimate within that group.
"""
(PROJECT_ROOT / "docs" / "passenger_survey_harmonization_note.md").write_text(methods_note, encoding="utf-8")

print("Wrote:")
for output_name in [
    "survey_availability_matrix.csv", "survey_category_crosswalk.csv",
    "survey_route_crosswalk.csv", "survey_harmonized_combined.csv",
    "survey_percentage_sum_checks.csv",
]:
    print(" -", PROCESSED / output_name)
print(" -", PROJECT_ROOT / "docs" / "passenger_survey_harmonization_note.md")
print("\nRoute match statuses:")
print(route_crosswalk["match_status"].value_counts())
print("\nAvailability across matched routes:")
display(availability_matrix.groupby("characteristic")[["available_2015_17", "available_2025", "available_both"]].sum())
print("\nPercentage checks:")
print(percentage_sum_checks["check_status"].value_counts(dropna=False))
print("\nCombined rows by survey period:")
print(combined.groupby("survey_period").size())

In [ ]:
# Keep source provenance explicit in the generated methods note.
note_path = PROJECT_ROOT / "docs" / "passenger_survey_harmonization_note.md"
note_text = note_path.read_text(encoding="utf-8")
note_text = note_text.replace(
    "`source_values_original`. Old values are weighted estimates",
    "`source_values_original`, and `source_route_values_original` for route-level components when records are summed. Old values are weighted estimates",
)
if "## Sources" not in note_text:
    note_text += "\n\n## Sources\n\n- 2015-17 CTPS survey: https://www.ctps.org/dv/mbtasurvey2018/index.html\n- 2025 MassDOT survey: https://gis.data.mass.gov/datasets/MassDOT::mbta-2025-system-wide-passenger-survey\n"
note_path.write_text(note_text, encoding="utf-8")
print("Updated methods note:", note_path)